# 🔬 Vigyan-3B-4x-MoE: Multi-Expert STEM Reasoning Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyansh001boy-tech/vigyan-7b-demo/blob/main/vigyan_3b_moe_demo.ipynb)
&nbsp;[![License: CC BY-NC 4.0](https://img.shields.io/badge/License-CC_BY--NC_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
&nbsp;[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Vigyan--3B--4x--MoE-yellow)](https://huggingface.co/shreyansh12183/Vigyan-3B-4x-MoE)

---

### 🌟 Model Overview
**Vigyan-3B-4x-MoE** is a 4-expert sparse Mixture of Experts upcycled from  across Science, Technology, Engineering, and Mathematics expert router heads.
- **Hardware:** Free Google Colab **Tesla T4 GPU** (uses ~3.2 GB VRAM in 4-bit NF4).
- **License:** CC BY-NC 4.0 (Non-commercial educational & research use).

In [ ]:
!pip install -q transformers accelerate bitsandbytes gradio
print("✅ Runtime packages installed.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "shreyansh12183/Vigyan-3B-4x-MoE"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

print(f"📥 Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
print("✓ Model loaded successfully!")

In [ ]:
import gradio as gr

def chat(message, history):
    prompt = f"<|im_start|>user
{message}<|im_end|>
<|im_start|>assistant
"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=350, temperature=0.2, do_sample=True)
    return tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

demo = gr.ChatInterface(fn=chat, title="🔬 Vigyan-3B-4x-MoE Playground", description="4-Expert STEM Mixture of Experts | Created by Shreyansh Singh")
demo.launch(share=True)